# Ames Housing Price Model — Training & Deployment Pipeline

This notebook trains the model that powers our house-price predictions and packages it for deployment on AWS. It takes the raw Ames, Iowa sales data, learns the relationship between a home's attributes and its sale price, and produces a single portable model file.

The exploratory work — comparing candidate algorithms (linear models, random forest, gradient boosting) and tuning them — lives in separate research notebooks. **This notebook is the conclusion of that work:** it rebuilds the winning model in a clean, reproducible way and prepares it to be served in production.

**Pipeline at a glance:** raw data → preprocessing → model training → validation → serialized artifact → AWS (S3 + Lambda).

## 1. Data loading and feature typing

We load the raw dataset and correct how a few columns are interpreted.

Several columns store numbers that aren't really quantities — a year like `2007`, or a building-class code like `60`. Left as numbers, the model would wrongly assume `2007` is "one unit better" than `2006`. We recast them to categories so each value is treated as a distinct label.

We also drop a handful of square-footage columns whose information is already captured by aggregate totals (e.g. total living area), so the model isn't fed the same signal twice.

In [ ]:
## Cell 1 — load + dtype casting + feature lists

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor

DATA_PATH = "Ames_Housing_Price_Data.csv"   # Stage 1: this line becomes an S3 download

df = pd.read_csv(DATA_PATH, index_col=0).drop(columns=["PID"])
for c in ["YrSold", "MoSold", "YearRemodAdd", "YearBuilt", "MSSubClass"]:
    df[c] = df[c].astype("object")
df["GarageYrBlt"] = df["GarageYrBlt"].astype("str")

y = df["SalePrice"]
X = df.drop(columns="SalePrice")

drop_num = ["BsmtFinSF2", "BsmtUnfSF", "BsmtFinSF1", "1stFlrSF", "2ndFlrSF", "LowQualFinSF"]
numeric_features = X.select_dtypes(include=["int64", "float64"]).drop(columns=drop_num).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.difference(["Electrical"])

## 2. Feature taxonomy: ordinal vs. nominal

Categorical features come in two kinds, and handling them correctly matters:

- **Ordinal** features have a meaningful order. A kitchen rated `Excellent` genuinely outranks `Good`, which outranks `Fair`. We encode these as ranked numbers so the model can use that order.
- **Nominal** features have no inherent ranking — a neighborhood or roof style is just a label. These get one-hot encoded (a separate yes/no column per category) so no false ordering is implied.

The dictionary below records the correct rank for every ordinal feature, taken from the dataset's data dictionary.

In [ ]:
## Cell 2 — ordinal map + feature split

ordinal_order = {
    "Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageFinish": ["None", "Unf", "RFn", "Fin"],
    "GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "PavedDrive": ["N", "P", "Y"],
    "PoolQC": ["None", "Fa", "TA", "Gd", "Ex"],
    "Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"],
}
ord_feats = [f for f in ordinal_order if f != "Electrical"]
nominal_features = [f for f in categorical_features if f not in ordinal_order]

## 3. Preprocessing pipeline

Raw data isn't model-ready: it has missing values, text categories, and numeric features on wildly different scales. This step bundles every transformation into one object, so the **exact same preprocessing is applied during training and later in production** — eliminating one of the most common deployment bugs.

Each feature group is handled appropriately: missing values filled, ordinal features ranked, nominal features one-hot encoded, numeric features standardized. The `handle_unknown` settings are a deliberate production safeguard — if a live request ever contains a category the model never saw in training, the pipeline handles it gracefully instead of crashing.

In [ ]:
## Cell 3 — the four transformers + combined preprocessor

electrical = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(categories=[ordinal_order["Electrical"]],
                           handle_unknown="use_encoded_value", unknown_value=-1)),
])
numeric = Pipeline([
    ("imp", SimpleImputer(strategy="mean")),
    ("sc", StandardScaler()),
])
ordinal = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="None")),
    ("ord", OrdinalEncoder(categories=[ordinal_order[f] for f in ord_feats],
                           handle_unknown="use_encoded_value", unknown_value=-1)),
])
nominal = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="None")),
    ("oh", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("electrical", electrical, ["Electrical"]),
    ("num", numeric, numeric_features),
    ("ord", ordinal, ord_feats),
    ("nom", nominal, nominal_features),
])

# The handle_unknown settings on the ordinal encoders are the one substantive change from the originals.
# Without them, a category the model never saw during training crashes the encoder: fine in a notebook,
# but a 500 error in Lambda.

## 4. Model and validation

We train a **Gradient Boosting** model — an ensemble that builds many small decision trees in sequence, each correcting the errors of the last. It was the strongest performer in our earlier model comparison.

To judge how well it generalizes to unseen homes, we use **10-fold cross-validation**: the data is split into ten parts, and the model is trained and tested ten times, each time holding out a different part. The reported **R²** is the average across all ten folds — it's the share of price variation the model explains, where closer to 1.0 is better.

In [ ]:
## Cell 4 — full model + a quick sanity check

model = Pipeline([
    ("pre", preprocessor),
    ("reg", GradientBoostingRegressor(max_depth=9, max_features="sqrt",
                                      min_samples_leaf=4, n_estimators=190,
                                      random_state=42)),
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, scoring="r2", cv=kf, n_jobs=-1)
print(f"10-fold R2: mean={scores.mean():.4f}  std={scores.std():.4f}")

## 5. Building the deployment artifact

With performance confirmed, we fit the model on the **entire** dataset — for the model we actually ship, we want it to learn from every available example — and serialize it to a single file with `joblib`.

This `.joblib` file is the deliverable. It holds the full pipeline (preprocessing **and** the trained model), so anything that loads it can turn a raw house record straight into a price. In the next stage, this file is uploaded to AWS S3.

In [ ]:
## Cell 5 — fit on all data + save the artifact

model.fit(X, y)
joblib.dump(model, "ames_model.joblib")   # Stage 1: this file gets uploaded to S3

## 6. What drives price — model interpretation

A prediction is more trustworthy when we can explain it. Here we measure **feature importance** — how much each attribute contributes to the model's predictions — and average it across the cross-validation folds, so the ranking reflects a stable pattern rather than one lucky split.

This is the explainability layer: it lets us tell a stakeholder *why* a home is valued the way it is, in plain terms (overall quality, living area, garage size, and so on).

In [ ]:
## Cell 6 — feature importance across folds

# Refit on each fold and collect the model's feature importances, so the final ranking
# is averaged over 10 splits rather than dependent on a single one.
feature_importances = []          # NOTE: this list was never initialized in the original (caused a NameError)

for train_index, test_index in kf.split(X):
    X_train, y_train = X.iloc[train_index], y.iloc[train_index]
    model.fit(X_train, y_train)

    importances = model.named_steps["reg"].feature_importances_

    # Rebuild feature names in the SAME order the ColumnTransformer outputs them:
    # electrical -> numeric -> ordinal -> one-hot nominal
    onehot_features = (model.named_steps["pre"]
                            .named_transformers_["nom"]
                            .named_steps["oh"].get_feature_names_out())
    feature_names = ["Electrical"] + list(numeric_features) + list(ord_feats) + list(onehot_features)

    if len(feature_names) != len(importances):
        print(f"Mismatch: {len(feature_names)} names vs {len(importances)} importances")
        continue

    feature_importances.append(pd.DataFrame({"Feature": feature_names, "Importance": importances}))

In [ ]:
## Cell 7 — aggregate importances across folds

all_importances = pd.concat(feature_importances)
aggregated_importances = (all_importances
                          .groupby("Feature").mean()
                          .reset_index()
                          .sort_values("Importance", ascending=False))

# NOTE: removed the original `.div(5)` line. groupby().mean() already averages across the
# folds, so importances correctly sum to ~1.0; dividing again distorted the values.
aggregated_importances.head(10)

In [ ]:
## Cell 8 — visualize the top drivers

import matplotlib.pyplot as plt

top_features = aggregated_importances.head(25)   # top 25 drivers
plt.figure(figsize=(10, 8))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Average Importance")
plt.ylabel("Features")
plt.title("What Drives Sale Price — Aggregated Feature Importance (10-fold)")
plt.gca().invert_yaxis()
plt.show()

## 7. The inference contract

Finally, we reload the saved model from disk and run a single prediction — **exactly the sequence the AWS Lambda function will perform for every incoming request:** load the model, accept one raw house record, return a price.

Proving it works here means deployment is just a matter of moving this same logic to the cloud.

In [ ]:
## Cell 9 — the inference contract (this becomes the Lambda handler)

loaded = joblib.load("ames_model.joblib")
one_row = X.iloc[[0]]                       # a single raw house record
print("predicted:", round(float(loaded.predict(one_row)[0])),
      " actual:", int(y.iloc[0]))

## STAGE 1: Uploading the Model as joblib. 
### Authenticating to AWS

Before touching S3, boto3 needs to prove who we are. Rather than placing access
keys in the notebook, we point boto3 at a **named profile** — credentials stored
in a file on this machine, outside the project folder, which boto3 reads
automatically. The notebook itself never contains anything secret.

The `get_caller_identity()` call is a free identity check: it confirms the
credentials load and shows which IAM user we're acting as, before we attempt
anything that could fail on permissions.

In [ ]:
## Stage 1 · Cell 2 — authenticate via a named profile (no keys in the notebook)
import boto3

session = boto3.Session(profile_name="The_Dev", region_name="us-east-1")

# Free sanity check: confirms the credentials load and shows which IAM user we are.
sts = session.client("sts")
print("authenticated as:", sts.get_caller_identity()["Arn"])

s3 = session.client("s3")
print("S3 client ready in:", s3.meta.region_name)

### Creating an S3 bucket

A **bucket** is a top-level storage container in S3 — think of it as the root of
a private cloud drive. Everything we store lives inside one.

Bucket names are **globally unique across all of AWS** (not just our account), so
the name below must be edited to something nobody else has taken. The `try/except`
makes this cell safe to re-run: if the bucket already exists and is ours, we just
reuse it instead of erroring.

In [ ]:
## Stage 1 · Cell 3 — create your bucket
# Rules: globally unique, lowercase, 3-63 chars, hyphens OK, NO underscores.
# Edit the suffix to something only you'd pick.
BUCKET = "ames-aws-update-2026"   # <-- edit this

from botocore.exceptions import ClientError
try:
    s3.create_bucket(Bucket=BUCKET)   # us-east-1 needs no extra location config
    print("created:", BUCKET)
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code == "BucketAlreadyOwnedByYou":
        print("already yours, reusing:", BUCKET)
    elif code == "BucketAlreadyExists":
        print("that name is taken globally — pick a different suffix")
    else:
        raise

### Uploading the model

We push `ames_model.joblib` — the artifact trained earlier in Step 7 — into
the bucket. Note that only the *model* goes to the cloud, not the CSV: the model
already learned everything it needs from the data, so the raw data stays local.

We then list the bucket's contents to confirm the file landed.

In [ ]:
## Stage 1 · Cell 4 — upload the model, then confirm it's there
s3.upload_file("ames_model.joblib", BUCKET, "ames_model.joblib")
print("uploaded ->", BUCKET)

for obj in s3.list_objects_v2(Bucket=BUCKET).get("Contents", []):
    print("  ", obj["Key"], f'({obj["Size"]/1e6:.2f} MB)')

### Round-trip verification

The real test: download the model *back* from S3 under a new filename, load it,
and make a prediction. If the price comes out, the model survived the trip to the
cloud and back intact.

This download-load-predict sequence is exactly what the Lambda function will do in
Stage 2 — the only difference is Lambda runs it on AWS's hardware instead of here.

In [ ]:
## Stage 1 · Cell 5 — download it back and confirm it still works
s3.download_file(BUCKET, "ames_model.joblib", "ames_model_from_s3.joblib")

import joblib
roundtrip = joblib.load("ames_model_from_s3.joblib")
print("prediction from the S3 copy:", round(float(roundtrip.predict(X.iloc[[0]])[0])))

## STAGE 2: Push — resolve the ECR address
Building the address from your account ID so nothing's typed from memory. That prints your exact tag and push commands with the real URI filled in — copy them into PowerShell.

In [ ]:
import boto3

session = boto3.Session(profile_name="The_Dev", region_name="us-east-1")
sts = session.client("sts")
ecr = session.client("ecr")

account  = sts.get_caller_identity()["Account"]
region   = session.region_name
repo     = "ames-lambda"

registry  = f"{account}.dkr.ecr.{region}.amazonaws.com"
image_uri = f"{registry}/{repo}:latest"

print("registry :", registry)
print("image URI:", image_uri)
print()
print("Run these in PowerShell after the login cell succeeds:")
print(f"  docker tag ames-lambda:latest {image_uri}")
print(f"  docker push {image_uri}")

### Authenticate Docker to ECR. 
Boto3 fetches the token, decodes it, and pipes it straight into docker login.

--password-stdin is deliberate — it pipes the password in instead of putting it in the command line, so the secret never lands in your shell history or the process list. You want Login Succeeded printed.

One thing that surprises people: logging in here authenticates Docker for your whole machine, including the PowerShell window. docker login writes to your user's Docker config, not to this Jupyter session — so even though boto3 did the login, the PowerShell docker push you run next is already authenticated. That's why the auth can happen in Python and the push in PowerShell.

In [ ]:
## Stage 2 · push — boto3 gets a temp token, hands it to Docker
import base64, subprocess

auth = ecr.get_authorization_token()["authorizationData"][0]
user, password = base64.b64decode(auth["authorizationToken"]).decode().split(":")
# the token decodes to "AWS:<temporary-password>" — valid ~12 hours

res = subprocess.run(
    ["docker", "login", "--username", user, "--password-stdin", registry],
    input=password, text=True, capture_output=True,
)
print(res.stdout.strip() or res.stderr.strip())   # want: "Login Succeeded"

In [ ]:
### Then, in PowerShell — the two commands Cell 1 printed:

# docker tag ames-lambda:latest <paste-image-uri-from-cell-1>
    
# docker push <paste-image-uri-from-cell-1>

## STAGE 3: Serve it over HTTPS — API Gateway + IAM auth

Stage 2 left us with a Lambda function (`ames-predictor`) that runs the container. Callers can reach it only through the AWS SDK (`lambda.invoke`). This stage puts a real HTTPS endpoint in front of it, `POST /predict`, and locks that endpoint with **IAM auth (SigV4)**.

**Why IAM and not Cognito/JWT or an API key?** The consumers are internal: teammates and internal systems that already hold AWS identities. Reusing IAM means no separate identity provider to stand up and no new credential type to manage. The trade-off is deliberate: callers still need AWS credentials. IAM auth moves that requirement to a new door; it doesn't remove it.

### One-time console setup (done as admin)

**1. Create the HTTP API**
- API Gateway → *Create API* → **HTTP API** → *Build*
- Integration: **Lambda** → `us-east-1` → `ames-predictor`, payload format **2.0**. Name: `ames-api`.
- Route: `POST /predict` → the `ames-predictor` integration.
- Stage: the default `$default` stage with auto-deploy (HTTP APIs have no manual Deploy step).
- Accept the prompt to add the resource-based permission that lets API Gateway invoke the function.
- *Develop → Authorization* → select `POST /predict` → set authorization to **AWS IAM**. The route is never public, even briefly.

**2. Grant the caller permission to invoke this one route**

A customer-managed policy, `ames-api-invoke`, attached to the `Ames_Project` group (which `The_Dev` belongs to):

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": "execute-api:Invoke",
    "Resource": "arn:aws:execute-api:us-east-1:<account-id>:<api-id>/*/POST/predict"
  }]
}
```

Reading the ARN: this API, any stage (`*`), `POST`, `/predict`, and nothing else.

**Two separate principals per request:** the *caller* (`The_Dev`) needs `execute-api:Invoke`, and the *Lambda execution role* reads the model from S3.

### Why `app.py` handles two call styles

With payload format 2.0, API Gateway delivers the request body as a **JSON string** in `event["body"]`, not as `event["features"]`. On the way out, because the handler's return value has a `statusCode`, API Gateway treats it as a full proxy response and expects `headers` plus a stringified `body`. So `app.py`:

- **unwraps** `event["body"]` when it's present (direct invokes still pass the payload as-is), and
- **wraps** every reply as `{statusCode, headers, body}`.

After rebuilding the image (`docker buildx build --platform linux/amd64 --provenance=false --sbom=false --output type=docker -t ames-lambda:latest .`) and pushing it as in Stage 2, the function has to be **explicitly pointed at the new image**. Pushing a new `:latest` doesn't change the running function by itself.

In [ ]:
## Stage 3 · Cell 1 — a JSON-safe canary record
# to_json serializes numpy dtypes and turns NaN into null; prep_raw maps null back to NaN for the imputers.
import json
row0 = json.loads(X.iloc[[0]].to_json(orient="records"))[0]
print("actual sale price:", int(y.iloc[0]))

In [ ]:
## Stage 3 · Cell 2 — point the function at the new image and wait for the update to finish
import boto3

session = boto3.Session(profile_name="The_Dev", region_name="us-east-1")
lam = session.client("lambda")

account = session.client("sts").get_caller_identity()["Account"]
ECR_URI = f"{account}.dkr.ecr.us-east-1.amazonaws.com/ames-lambda:latest"

lam.update_function_code(FunctionName="ames-predictor", ImageUri=ECR_URI, Publish=True)
# update_function_code returns immediately; Lambda re-pulls the image in the background.
lam.get_waiter("function_updated_v2").wait(FunctionName="ames-predictor")

info = lam.get_function_configuration(FunctionName="ames-predictor")
print("update status:", info["LastUpdateStatus"])   # want: Successful

In [ ]:
## Stage 3 · Cell 3 — direct invoke in the cloud (isolates one variable: did the new image deploy?)
resp = lam.invoke(FunctionName="ames-predictor",
                  Payload=json.dumps({"features": row0}).encode("utf-8"))
outer = json.loads(resp["Payload"].read())
print("price:", json.loads(outer["body"])["predicted_price"])   # want: 126629.23

### The real end-to-end: a SigV4-signed POST

This time we sign the request by hand instead of letting the SDK do it invisibly. `profile_name="The_Dev"` is a **label, not a password**: it points boto3 at the `[The_Dev]` section of the local credentials file (`~/.aws/credentials`), which lives outside this project. `SigV4Auth.add_auth()` uses the secret key to compute a signature over this exact request (method, host, path, body, timestamp) and adds it as headers. Only that signature goes over the network, never the key itself. API Gateway recomputes the signature, then checks that the caller's IAM policy allows `POST /predict`.

In [ ]:
## Stage 3 · Cell 4 — find the API's invoke URL (no IDs hard-coded)
API_NAME = "ames-api"   # <-- edit if you named the API differently

apigw = session.client("apigatewayv2")
apis = {a["Name"]: a for a in apigw.get_apis()["Items"]}
URL = apis[API_NAME]["ApiEndpoint"] + "/predict"
print("endpoint:", URL)

In [ ]:
## Stage 3 · Cell 5 — signed request through API Gateway (should succeed)
import requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

creds = session.get_credentials().get_frozen_credentials()
body = json.dumps({"features": row0})   # the SAME body is signed and sent; a mismatch -> 403

req = AWSRequest(method="POST", url=URL, data=body,
                 headers={"Content-Type": "application/json"})
SigV4Auth(creds, "execute-api", "us-east-1").add_auth(req)   # adds Authorization + X-Amz-Date

resp = requests.post(URL, data=body, headers=dict(req.headers))
print("HTTP status:", resp.status_code)   # want: 200
print("response:", resp.json())           # want: {'predicted_price': 126629.23}

API Gateway reads the `{statusCode, headers, body}` shape from `app.py` and turns it into a real HTTP response. That's why the client gets a clean `{'predicted_price': ...}` here, while the direct invoke in Cell 3 had to dig the price out of `body`.

### The negative test: prove the lock works

The same request with the signing step removed is how an outsider with no AWS credentials would hit the endpoint. IAM auth refuses it with **403 Forbidden**. (A Cognito/JWT authorizer would have returned 401 instead.)

In [ ]:
## Stage 3 · Cell 6 — unsigned request (should be refused)
resp = requests.post(URL, data=body, headers={"Content-Type": "application/json"})
print("HTTP status:", resp.status_code)   # want: 403
print("response:", resp.json())           # want: {'message': 'Forbidden'}

### Result

The same canonical prediction, **126629.23**, came back on four independent paths: the local container (Lambda RIE), the S3 round trip, a direct cloud invoke, and a SigV4-signed HTTPS POST through API Gateway. An unsigned request is refused with 403. The deployment is complete: a frozen pipeline in a container on Lambda, fed by S3, behind an HTTPS endpoint that only authorized IAM identities can call.